# MiniGPT: Transformer & GPT from Scratch (Google Colab)

This notebook runs the whole assignment end to end on Google Colab with a GPU: setup, unit tests, overfit gate, full training, loss curves, generation, attention inspection, evaluation, ablations, and inference. The project and results are stored on Google Drive so they survive runtime disconnects.

**Before you start**
1. Push your project to a GitHub repository (see `GUIDE.md`, Step 8), or upload the zip (cell 2b).
2. Run the setup cell and authorize Google Drive when prompted.
3. Set `REPO_URL` in the setup cell to your repository.
4. Runtime -> Change runtime type -> **T4 GPU** (recommended; CPU also works but is slow).
5. Runtime -> Run all. Full training takes roughly 10-25 minutes on a T4.

If Colab disconnects, reconnect, rerun setup, and continue from the last completed section. Checkpoints and outputs are kept in `MyDrive/MiniGPT/`.


## 1. Get the code

In [ ]:
from pathlib import Path
import os

from google.colab import drive
drive.mount("/content/drive")

REPO_URL = "https://github.com/YOUR-USERNAME/transformer-gpt-assignment.git"  # <-- change me
DRIVE_ROOT = Path("/content/drive/MyDrive/MiniGPT")
REPO_DIR = DRIVE_ROOT / "transformer-gpt-assignment"
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

if not (REPO_DIR / "requirements.txt").exists():
    if "YOUR-USERNAME" in REPO_URL:
        raise ValueError("Set REPO_URL to your GitHub repository before running this cell.")
    !git clone -q "$REPO_URL" "$REPO_DIR"
os.chdir(REPO_DIR)
print("persistent project directory:", Path.cwd())
print("results directory:", REPO_DIR / "outputs")


**2b. Alternative (no GitHub):** run this cell instead of cell 1 and upload `transformer-gpt-assignment.zip`.

In [ ]:
# from google.colab import files
# import zipfile
# up = files.upload()                       # choose the zip
# zip_path = next(iter(up))
# zipfile.ZipFile(zip_path).extractall(DRIVE_ROOT)
# os.chdir(REPO_DIR)


## 2. Install dependencies and check the GPU

In [ ]:
%pip install -q -r requirements.txt
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: select Runtime -> Change runtime type -> T4 GPU for faster training.")
print("project:", os.getcwd())


## 3. Unit tests (architecture and causal-mask checks)

In [ ]:
!python -m pytest -q

## 4. Overfit gate (Milestone 4): the loss must fall substantially on a tiny batch

In [ ]:
!python -m src.train --overfit --run-name overfit --device auto

## 5. Full MiniGPT training (Milestone 5)

In [ ]:
!python -m src.train --config configs/tiny_gpt.yaml --run-name mini_gpt --device auto

## 6. Loss curves (Figure 6)

In [ ]:
!python -m experiments.plot_curves --run mini_gpt
from IPython.display import Image, display
display(Image("outputs/figures/fig06_loss_curves.png"))
!cat outputs/logs/mini_gpt/summary.json


## 7. Generate samples from fixed prompts (Figure 8)

In [ ]:
from pathlib import Path
from src.generate import load_model, generate_text
from src.utils import count_parameters, get_device

checkpoint = Path("checkpoints/mini_gpt_best.pt")
if not checkpoint.exists():
    raise FileNotFoundError(f"Missing {checkpoint}. Run training first or restore the Drive project.")
device = get_device("auto")
model, tok = load_model(str(checkpoint), device)
print(f"Loaded {checkpoint} on {device}; parameters: {count_parameters(model):,}")
for prompt in ["ROMEO:", "First Citizen:\n", "To be, or not to be", "KING HENRY:", "The future of"]:
    print("=" * 70)
    print(generate_text(model, tok, prompt, max_new_tokens=120, temperature=0.8, top_k=40, seed=0, device=device))


## 8. Hand-calculation figures, mask, positional encoding, shape trace, model summary (Figures 2-5)

In [ ]:
!python -m experiments.static_figures
from IPython.display import Image, display
display(Image("outputs/figures/fig02_causal_mask.png"))
display(Image("outputs/figures/fig04_positional_encoding.png"))


## 9. Attention inspection of the trained model (Figure 7)

In [ ]:
!python -m experiments.attention_visualization --prompt "ROMEO:"
from IPython.display import Image, display
display(Image("outputs/figures/fig07_attention_trained.png"))


## 10. Evaluation: perplexity, causality check, samples

In [ ]:
!python -m experiments.evaluate
!head -n 30 outputs/samples/eval_samples.md


## 11. Ablation studies (Figure 9). Slow: about 12 short training runs. Lower `--steps` for a quick pass.

In [ ]:
!python -m experiments.ablation --steps 1500


## 12. Inference CLI (Figure 10 evidence)

In [ ]:
from pathlib import Path
checkpoint = Path("checkpoints/mini_gpt_best.pt")
if not checkpoint.exists():
    raise FileNotFoundError(f"Missing {checkpoint}. Run the training cell first.")
!python -m src.generate --ckpt checkpoints/mini_gpt_best.pt --prompt "The future of" --max-new-tokens 80 --temperature 0.8 --device auto


## 13. Bundle all results for download

In [ ]:
from pathlib import Path
import zipfile

bundle_path = Path("/content/drive/MyDrive/MiniGPT/results_bundle.zip")
with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as bundle:
    for path in Path("outputs").rglob("*"):
        if path.is_file():
            bundle.write(path, path.as_posix())
    checkpoint = Path("checkpoints/mini_gpt_best.pt")
    if checkpoint.exists():
        bundle.write(checkpoint, checkpoint.as_posix())
print("Saved results:", bundle_path)
print("Checkpoint:", Path("checkpoints/mini_gpt_best.pt").resolve())
try:
    from google.colab import files
    files.download(str(bundle_path))
except ImportError:
    print("Not on Colab: results are saved in Google Drive.")
